# LS1 — Qwen2.5-VL-3B-Instruct

Offline Kaggle T4, independent exploratory scorer. Add Input: the reviewed LS1 source package, existing model prep output and matching runtime prep output (InternVL3 FIX v2 where applicable). Set Internet OFF; the child sees one T4. No dependencies or weights are downloaded.

Edit only SETTINGS: reviewed source SHA, source archive checksum, trusted runtime inventory checksum and Internet confirmation. Attach the supplementary runtime inventory from source preparation. Empty attachment paths request unambiguous discovery; specify input-relative paths when names differ or several inputs match. Run All once. Existing attempts are never resumed or overwritten.

Default technical mode uses four synthetic colored drawings outside InspecSafe, plus three fixed calibration images, with C1 unchanged. It tests plumbing and makes no accuracy claim. Train and test are LOCKED: changing mode alone cannot authorize execution. They require a Research Lead approval receipt and trusted SHA bound to this exact split/run/source/method. See notes/ls1_runbook.md for approval and artifact auditing. Test remains locked by default.

In [ ]:
# One configuration cell; no manual edits to execution cells.
SETTINGS = {
    "model_key": "qwen2_5",
    "mode": "technical",
    "run_id": "ls1-qwen2_5-technical-first",
    "source_commit": "",
    "source_archive_sha256": "",
    "internet_off_confirmed": False,
    "source_release_input": "",
    "model_input": "",
    "runtime_input": "",
    "runtime_site_packages": "",
    "runtime_inventory_input": "",
    "runtime_inventory_sha256": "",
    "approval_input": "",
    "approval_sha256": "",
    "protect_level01": False,
    "dataset_root": "",
    "manifest_path": "",
    "provenance_path": ""
}


In [ ]:
# Verify the committed source package before importing any attached code.
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import subprocess
import sys
import tarfile

work = Path.cwd()
inputs = work.parent / "input"
repo = work / ("ls1-source-" + SETTINGS["run_id"])
sha = SETTINGS["source_commit"]
if len(sha) != 40 or any(c not in "0123456789abcdef" for c in sha):
    raise RuntimeError("Fill the exact reviewed Draft HEAD from source preparation")
if SETTINGS["internet_off_confirmed"] is not True:
    raise RuntimeError("Set Kaggle Internet OFF and confirm in SETTINGS")
if SETTINGS["source_release_input"]:
    rel = PurePosixPath(SETTINGS["source_release_input"])
    if rel.is_absolute() or ".." in rel.parts or "\\" in str(rel) or ":" in str(rel):
        raise RuntimeError("Source receipt must be input-relative")
    receipts = [inputs / rel]
else:
    receipts = list(inputs.glob("*/ls1_source_release.json"))
if len(receipts) != 1:
    raise RuntimeError("Attach exactly one LS1 source prep output, or set source_release_input")
receipt = json.loads(receipts[0].read_text())
if receipt.get("schema") != "ls1-source-release-v1" or receipt.get("source_commit") != sha:
    raise RuntimeError("Source release identity mismatch")
if receipt.get("archive") != "ls1_source.tar" or receipt.get("repository") != "tantdna2/SafeShift":
    raise RuntimeError("Source archive receipt mismatch")
archive = receipts[0].parent / receipt["archive"]
raw = archive.read_bytes()
if hashlib.sha256(raw).hexdigest() != receipt["archive_sha256"]:
    raise RuntimeError("Source archive checksum mismatch")
if receipt["archive_sha256"] != SETTINGS["source_archive_sha256"]:
    raise RuntimeError("Fill the independently verified source archive SHA-256")
commit = subprocess.check_output(["git", "get-tar-commit-id"], input=raw).decode().strip()
if commit != sha:
    raise RuntimeError("Git archive embedded commit mismatch")
repo.mkdir(exist_ok=False)
with tarfile.open(archive, "r:") as stream:
    members = stream.getmembers()
    names = set()
    for member in members:
        path = PurePosixPath(member.name)
        if (path.is_absolute() or ".." in path.parts or "\\" in member.name or ":" in member.name
                or member.name in names or not (member.isfile() or member.isdir())):
            raise RuntimeError("Unsafe or duplicate source archive entry")
        names.add(member.name)
    stream.extractall(repo, members=members, filter="data")
sys.path.insert(0, str(repo))
os.environ.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
                   "HF_HUB_DISABLE_TELEMETRY": "1", "HF_HUB_DISABLE_XET": "1",
                   "HF_HUB_ENABLE_HF_TRANSFER": "0", "CUDA_VISIBLE_DEVICES": "0"})
from safeshift.ls1.kaggle import launch
from safeshift.runners.internvl3_snapshot import network_denied
SETTINGS["source_receipt"] = receipt
with network_denied():
    output = launch(repo, inputs, SETTINGS)
print("LS1 artifacts:", output)
print("Technical mode establishes no performance result. Keep historical runs unchanged.")
